# C1.3 · Attacking evaluation itself

**Function C — Red Teaming and Security Research with AI → Red Teaming with AI**  ·  *Security of AI*

Builds on **[C1.2 · Red-teaming an agent: designing the campaign](https://spbreed.github.io/cyber-commons/lessons/C1.2.html)**.

| | |
|---|---|
| Tools used | Cyber Commons eval harness, Kimi K2, Claude Haiku 4.5 |

## What this lesson is

**What it covers.** Game the B2.1 harness deliberately, then close the hole you used.

**Why a security engineer needs it.** If the eval can be fooled, the assurance is theatre. The control it builds is: eval gaming, sandbagging, contamination and judge manipulation as test cases.

This is a **control** lesson: it builds the mechanism, then breaks it, so you can see what the control is actually load-bearing for rather than taking the claim on trust.

## 1 · The hook

Your evaluation is a control, and controls get attacked. A benchmark with a leaked key, a skewed class balance or a scorer that can be satisfied without solving anything is a control that reports itself green forever.

> **At CyberTravels.** The benchmark that says CyberTravels' review harness scores 0.9 is itself a control, and it gets attacked. A leaked key or a loose matcher makes it report green forever.

## 2 · The framework

```
   the benchmark is a control, so attack it

   leaked key      -> the score is a training metric
   skewed classes  -> a constant answer scores 0.875
   loose matching  -> wrong answers match on basename
   gameable oracle -> satisfied without solving anything

   an unattacked evaluation reports itself green forever
```

If you can make a harness score well without being good, so can the vendor whose
benchmark you are reading — and so can your own team, without meaning to.

Three exploits work on almost every published security-harness result:

1. **Report conformance as quality.** Schema validity is ~100% by construction
   with structured output. It measures nothing about correctness (B2.1).
2. **Exploit class imbalance.** If 80% of a corpus is one CWE, always guessing
   that CWE scores 0.8 with no capability at all.
3. **Exploit basename collisions.** If the matcher compares bare filenames and
   the corpus reuses `1.py` across directories, scores become partly random —
   and random noise on a leaderboard looks like a small improvement.

Red-teaming evaluation means running these three against your own numbers before
someone else does.

## 3 · The procedure, as a skill

Before believing an evaluation, score a harness with no capability at all. The skill does exactly that against the skewed corpus, then rebalances and re-scores — and separately checks whether the matcher rewards answers naming the wrong directory.

In [ ]:
# skills/redteam/eval-corpus-integrity-check/SKILL.md — embedded verbatim from the repository.
# This is the file itself, not a paraphrase of it.
SKILL_MD = r"""---
name: eval-corpus-integrity-check
description: >-
  Attack an evaluation rather than a system: score a deliberately
  zero-capability harness against the corpus and see what the corpus alone
  awards it, then rebalance and re-score. Use before trusting any benchmark
  number, your own included.
allowed-tools: Read, Grep, Glob
---

# Score the null harness first

An evaluation is a claim about a system, and it is only as good as the corpus
underneath it. The cheapest way to find out is to score a harness that has no
capability at all — one that answers from the corpus's own skew. A high score
there is a measurement of the corpus, and every number derived from it inherits
the problem.

## When to use this

Before publishing an evaluation, before believing one, and whenever a benchmark
result is surprisingly good.

## Procedure

**1 — Build the null harness.** No analysis: answer with whatever the corpus's
majority class is. It should be trivial to write, and writing it takes minutes.

**2 — Score it against the corpus as it stands.** Record conformance and
accuracy separately — a null harness usually conforms perfectly, which is itself
worth knowing about conformance as a metric.

**3 — Rebalance and re-score.** Equalise the classes and run the same null
harness. The drop is the portion of the original score that came from skew
rather than capability.

**4 — Check the matcher, not just the corpus.** Score answers that name the
wrong directory. If they still score, the matcher is measuring string similarity
rather than correctness, and it will flatter every harness equally.

**5 — Report the null score alongside every real one.** A harness scoring 0.85
against a null of 0.85 has demonstrated nothing, and that comparison is the only
honest way to read the number.

## Output contract

```json
{
  "corpus": {"n": 0, "skew": 0.0, "classes": {"str": 0}},
  "null_harness": {"conformance": 0.0, "accuracy_skewed": 0.0, "accuracy_balanced": 0.0},
  "matcher": {"kind": "str", "wrong_directory_scores": 0.0},
  "verdict": {"score_from_skew": 0.0, "score_from_capability": 0.0}
}
```

## Failure modes

- **Reporting accuracy without the null baseline.** It is unreadable without
  one.
- **Conflating conformance with accuracy.** The null harness conforms.
- **Trusting a matcher that has never been attacked.** Try to fool it before you
  publish with it.
"""

In [ ]:
import json, re

def parse_skill(md):
    """Split a SKILL.md into (frontmatter dict, body).

    Frontmatter is a small, fixed subset of YAML: `key: value`, plus folded
    scalars (`description: >-`) whose continuation lines are indented. That is
    all a skill needs, and parsing it directly means no dependency.
    """
    if not md.startswith("---"):
        raise ValueError("a SKILL.md must open with a frontmatter block")
    _, front, body = md.split("---", 2)
    meta, key = {}, None
    for line in front.strip().splitlines():
        if not line.strip():
            continue
        if not line[0].isspace() and ":" in line:
            key, val = line.split(":", 1)
            key, val = key.strip(), val.strip()
            # `>-` and `|` open a folded block; the value is on the next lines
            meta[key] = "" if val in (">-", ">", "|", "|-") else val
        elif key is not None:
            meta[key] = (meta[key] + " " + line.strip()).strip()
    if "allowed-tools" in meta:
        meta["allowed-tools"] = [t.strip() for t in meta["allowed-tools"].split(",")
                                 if t.strip()]
    for required in ("name", "description"):
        if not meta.get(required):
            raise ValueError(f"skill is missing a {required!r}")
    return meta, body.strip()

_WORD = re.compile(r"[a-z][a-z-]{3,}")

def route(task, skills):
    """Pick the skill whose description best matches a task. Deterministic.

    The description is not documentation — it is the routing key. An agent
    decides whether to load a skill by reading it, so a vague description means
    the skill never fires when it should, and two overlapping descriptions mean
    the wrong one fires.

    Returns (pick, scores, margin). A margin of 0 means the top two scored the
    same and the "winner" is just whichever sorted first — an arbitrary answer
    wearing a confident face. Callers should refuse to auto-route on margin 0
    rather than pretend the tiebreak meant something.
    """
    want = set(_WORD.findall(task.lower()))
    def score(meta):
        return len(want & set(_WORD.findall(meta["description"].lower())))
    scores = {n: score(skills[n]) for n in sorted(skills)}
    # sort names first, then by score: ties must break identically on every
    # machine or the same task routes differently on two runs
    ranked = sorted(sorted(skills), key=lambda n: -scores[n])
    top = scores[ranked[0]]
    margin = top - (scores[ranked[1]] if len(ranked) > 1 else 0)
    return ranked[0], scores, margin

def contract_of(body):
    """The JSON block under '## Output contract' — the skill's machine promise."""
    # non-greedy across any prose between the heading and the fence
    m = re.search(r"## Output contract\b.*?```json\n(.*?)```", body, re.S)
    if not m:
        raise ValueError("skill declares no output contract")
    return json.loads(m.group(1))

def check(instance, contract, path="$"):
    """Structural conformance of an instance against a contract template.

    Returns the list of problems. An empty list means the shape is right — and
    that is *all* it means. Conformance is not accuracy: an empty findings list
    conforms perfectly and tells you nothing.
    """
    problems = []
    if isinstance(contract, dict):
        if not isinstance(instance, dict):
            return [f"{path}: expected an object, got {type(instance).__name__}"]
        for k, v in sorted(contract.items()):
            if k not in instance:
                problems.append(f"{path}.{k}: missing")
            else:
                problems += check(instance[k], v, f"{path}.{k}")
    elif isinstance(contract, list):
        if not isinstance(instance, list):
            return [f"{path}: expected a list, got {type(instance).__name__}"]
        for i, item in enumerate(instance):          # every element, same template
            problems += check(item, contract[0], f"{path}[{i}]")
    elif isinstance(contract, str) and "|" in contract:
        if instance not in contract.split("|"):
            problems.append(f"{path}: {instance!r} is not one of {contract}")
    elif isinstance(contract, bool):                  # before the numeric case:
        if not isinstance(instance, bool):            # bool is a subclass of int
            problems.append(f"{path}: expected bool, got {type(instance).__name__}")
    elif isinstance(contract, (int, float)):
        # JSON has one number type. A contract written `0` must accept 0.4, or
        # every cost and rate in the pipeline has to be rounded to satisfy a
        # checker rather than to be correct.
        if isinstance(instance, bool) or not isinstance(instance, (int, float)):
            problems.append(f"{path}: expected a number, got {type(instance).__name__}")
    elif not isinstance(instance, type(contract)):
        problems.append(f"{path}: expected {type(contract).__name__}, "
                        f"got {type(instance).__name__}")
    return problems

# Execute the skill above: parse skills/redteam/eval-corpus-integrity-check/SKILL.md into the two
# halves an agent uses — the frontmatter it routes on, and the body
# it follows.
meta, body = parse_skill(SKILL_MD)
print(f"loaded skill: {meta['name']}")
print(f"  tools it may use: {', '.join(meta.get('allowed-tools', [])) or '—'}")
print(f"  routing description: {len(meta['description'].split())} words")
print(f"  procedure: {len(body.splitlines())} lines")

In [ ]:
# skills/redteam/eval-corpus-integrity-check/scripts/eval_corpus_integrity_check.py — embedded verbatim from the repository.
# This is the skill's own script, not a paraphrase of it.
#!/usr/bin/env python3
"""Score a zero-capability harness against a skewed corpus and again against a balanced one, and watch the number collapse.

This is the executable half of the `eval-corpus-integrity-check` skill: the check the
SKILL.md next to it describes, run against a synthetic CyberTravels
estate so two runs can be diffed and the result argued with.

Standard library only, and deterministic, so it runs on a Kaggle
kernel with the internet switched off.
"""

import json
from dataclasses import dataclass

@dataclass
class Truth:
    qid: str; cwe: str; file: str

def make_corpus(n=40, skew=0.8):
    """A corpus where `skew` of the answers are one class — very common."""
    truths = {}
    n_major = int(n * skew)
    for i in range(1, n + 1):
        cwe = "CWE-89" if i <= n_major else ["CWE-78", "CWE-22", "CWE-798"][i % 3]
        truths[f"q{i}"] = Truth(f"q{i}", cwe, f"{cwe}/{i}.py")
    return truths

SKEWED = make_corpus(40, skew=0.8)
from collections import Counter
print("class balance:", Counter(t.cwe for t in SKEWED.values()))

class NullHarness:
    """No capability whatsoever. Emits perfect JSON and always guesses CWE-89."""
    def answer(self, qid, truth):
        return json.dumps({"qid": qid, "cwe": "CWE-89", "file": truth.file,
                           "line": 1, "rationale": "user input is concatenated"})

null = NullHarness()
ANSWERS = {q: null.answer(q, t) for q, t in SKEWED.items()}

def path_key(p):
    parts = [x for x in p.replace("\\", "/").split("/") if x not in ("", ".")]
    return "/".join(parts[-2:]) if len(parts) > 1 else (parts[-1] if parts else "")
def basename(p): return p.replace("\\", "/").split("/")[-1]

def evaluate(answers, truths, matcher=path_key):
    conforming = expert = 0
    for qid, t in truths.items():
        try:
            d = json.loads(answers[qid])
        except json.JSONDecodeError:
            continue
        conforming += 1
        if matcher(d["file"]) != matcher(t.file):
            continue
        expert += 1.0 if d["cwe"].upper() == t.cwe else 0.5
    return {"conformance": conforming / len(truths),
            "expert_accuracy": expert / len(truths)}

r = evaluate(ANSWERS, SKEWED)
print("the null harness, scored on the skewed corpus:")
print(f"   conformance      {r['conformance']:.2f}   ← quotable as '100%'")
print(f"   expert accuracy  {r['expert_accuracy']:.2f}")
print("\nZero capability. Both numbers look like a working product.")

BALANCED = {}
for i in range(1, 41):
    cwe = ["CWE-89", "CWE-78", "CWE-22", "CWE-798"][i % 4]
    BALANCED[f"q{i}"] = Truth(f"q{i}", cwe, f"{cwe}/{i}.py")
ANS_B = {q: null.answer(q, t) for q, t in BALANCED.items()}

print("same null harness:")
for label, truths, ans in (("skewed corpus (80% CWE-89)", SKEWED, ANSWERS),
                           ("balanced corpus", BALANCED, ANS_B)):
    r = evaluate(ans, truths)
    print(f"   {label:28s} conformance {r['conformance']:.2f}  "
          f"expert accuracy {r['expert_accuracy']:.2f}")
print("\nBalancing the corpus removed most of the fake score. Nothing about the")
print("harness changed.")

# exploit 3: a matcher that compares bare filenames.
# Build answers that point at the WRONG directory but the right filename.
WRONG_DIR = {}
for q, t in BALANCED.items():
    n = q[1:]
    WRONG_DIR[q] = json.dumps({"qid": q, "cwe": t.cwe,
                               "file": f"CWE-89/{n}.py",     # wrong dir, right basename
                               "line": 1, "rationale": "untrusted input"})

for matcher, name in ((path_key, "path_key (parent + filename)"),
                      (basename, "basename only (the bug)")):
    r = evaluate(WRONG_DIR, BALANCED, matcher)
    print(f"{name:34s} expert accuracy {r['expert_accuracy']:.2f}")
print("\nEvery answer names the wrong file. The basename matcher scores them")
print("as correct, because the corpus reuses numeric filenames across directories.")

def audit_benchmark(truths, answers, matcher):
    from collections import Counter
    counts = Counter(t.cwe for t in truths.values())
    majority_share = max(counts.values()) / len(truths)
    always_majority = {q: json.dumps({"qid": q, "cwe": counts.most_common(1)[0][0],
                                      "file": t.file, "line": 1, "rationale": "x"})
                       for q, t in truths.items()}
    floor = evaluate(always_majority, truths, matcher)["expert_accuracy"]
    real  = evaluate(answers, truths, matcher)
    collisions = len(truths) - len({matcher(t.file) for t in truths.values()})
    return {
      "majority_class_share": round(majority_share, 2),
      "score_of_always_guessing_majority": round(floor, 2),
      "reported_expert_accuracy": round(real["expert_accuracy"], 2),
      "lift_over_trivial_baseline": round(real["expert_accuracy"] - floor, 2),
      "matcher_collisions": collisions,
      "conformance_reported_as_quality": real["expert_accuracy"] < real["conformance"] - 0.2,
    }

print("audit of the skewed benchmark with a basename matcher:")
for k, v in audit_benchmark(SKEWED, ANSWERS, basename).items():
    print(f"   {k:38s} {v}")
print("\naudit of the balanced benchmark with path_key:")
for k, v in audit_benchmark(BALANCED, ANS_B, path_key).items():
    print(f"   {k:38s} {v}")

a = audit_benchmark(BALANCED, ANS_B, path_key)
assert a["lift_over_trivial_baseline"] <= 0.01
print("\nThe null harness has ~zero lift over the trivial baseline, which is the")
print("only honest way to describe it.")

## What you just proved

On the skewed corpus the zero-capability harness scores conformance 1.00 and expert accuracy around 0.85. Balancing the corpus drops expert accuracy to roughly 0.25. Answers naming the wrong directory score near zero under `path_key` and near 1.00 under a basename matcher. The audit reports the majority-class share, the trivial baseline, the matcher collisions, and near-zero lift.

## Your turn

Run the audit against a benchmark result your organisation relies on. Two questions decide it: what does always guessing the majority class score, and does the matcher collide? Most published numbers answer neither.

---

**Next → [C1.4 · Reporting agentic findings](https://spbreed.github.io/cyber-commons/lessons/C1.4.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/C1.3.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/C1.3.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*